### Imports

In [2]:
from pathlib import Path
import json
import os 
import re

print("Imports Complete.")

Imports Complete.


### Inspect Directory

In [ ]:
current_directory = Path.cwd()               # returns the current working directory of this notebook
print("Current directory :")
print(current_directory)
print()
print("Files and Directories :")
for path in current_directory.iterdir():     # iterdir() will iterate over all the items in the directory
    print(path)

Current directory :
c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model

Files and Directories :
c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model\01_data_preparation.ipynb
c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model\data
c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model\README.md
c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model\tokenization_fundamentals


In [9]:
data_directory = current_directory / 'data'
print("Data directory :")
print(data_directory)
print()
print("Files and Directories")
for path in data_directory.iterdir():
    print(" ", path.name)

Data directory :
c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model\data

Files and Directories
  enwiki-2026-09-01-p80258756p84110898.jsonl.gz


In [11]:
corpus_path = data_directory / 'enwiki-2026-09-01-p80258756p84110898.jsonl.gz'
print("Corpus path :", corpus_path)
print("Exists      :", corpus_path.exists())
print("File name   :", corpus_path.name)
print("Suffixes    :", corpus_path.suffixes)

file_size_bytes = corpus_path.stat().st_size
file_size_mb = file_size_bytes / (1024 ** 2)

print(f"File size   : {file_size_mb:.2f} MB")

Corpus path : c:\Users\Shivanshu Sirohi\Documents\Multilingual_medical_BERT\stage_1_base_multilingual_model\data\enwiki-2026-09-01-p80258756p84110898.jsonl.gz
Exists      : True
File name   : enwiki-2026-09-01-p80258756p84110898.jsonl.gz
Suffixes    : ['.jsonl', '.gz']
File size   : 217.55 MB


In [13]:
import gzip                           # imports python library for reading/writing a .gz compressed file

with gzip.open(corpus_path, 'rt', encoding="utf-8") as file:       # open the compressed file directly in text reading mode
    first_line = file.readline()                                   # rt > r = read , t = text
                                                                   # readline reads only the first line/record from the corpus
    print("Raw line length :", len(first_line))
    print()
    print(first_line[:2000])



Raw line length : 1205

{"id":"enwiki_80258789","source":"wikipedia","wiki_id":"enwiki","snapshot":"2026-09-01","page_id":"80258789","title":"Ehsan Mehmood Khan","text":"is a retired two-star general of the Pakistan Army, who served as the Director General Infantry at the General Headquarters in Rawalpindi.\n\nEducation\nKhan has done his PhD in Peace and Conflict Studies.\n\nMilitary career\nKhan was commissioned into the 34 Punjab Regiment via the 22 OTS Course. In 2018, Khan was promoted to the rank Major General from Brigadier. As Force Commander Northern Areas, he oversaw military operations and civil-military initiatives in Gilgit-Baltistan. Later he served as Director General of the National Guard.\n\nHe also served as Director General Infantry at GHQ, Rawalpindi till retirement.\n\nHe was awarded the Hilal-i-Imtiaz by the then President of Pakistan in 2021 in recognition of his distinguished service.\n\nPublications\nKhan is the author of the book , which discusses non-traditio

In [14]:
## ^^ the text above is very well written and coherent. Exactly what we need to build our model

### Process JSON

In [18]:
first_record = json.loads(first_line)       # short for 'load strings' is used to parse a json formatted string and convert it into a corresponding python object
print("record type :", type(first_record))
print("fields      :", first_record.keys())
print()
for key, value in first_record.items():
    print(f"{key:10} | {type(value).__name__:8} | {repr(value)[:300]}")   # print key with a fixed width of 10 characters, value with a simple name like str instead of <class 'str'> and finally shows the value in debugging friendly format

record type : <class 'dict'>
fields      : dict_keys(['id', 'source', 'wiki_id', 'snapshot', 'page_id', 'title', 'text'])

id         | str      | 'enwiki_80258789'
source     | str      | 'wikipedia'
wiki_id    | str      | 'enwiki'
snapshot   | str      | '2026-09-01'
page_id    | str      | '80258789'
title      | str      | 'Ehsan Mehmood Khan'
text       | str      | 'is a retired two-star general of the Pakistan Army, who served as the Director General Infantry at the General Headquarters in Rawalpindi.\n\nEducation\nKhan has done his PhD in Peace and Conflict Studies.\n\nMilitary career\nKhan was commissioned into the 34 Punjab Regiment via the 22 OTS Course. 


In [20]:
num_records = 5

with gzip.open(corpus_path, 'rt', encoding='utf-8') as file:

    for i in range(num_records):
        line = file.readline()                        # readline keeps on reading line after line
        record = json.loads(line)

        print("=" * 30)
        print(f"Record : {i + 1}")
        print("=" * 30)

        print("ID     :", record['id'])
        print("Title  :", record['title'])
        print("Text lenght :", len(record['text']))
        print()
        print(record['text'][:500])
        print()


Record : 1
ID     : enwiki_80258789
Title  : Ehsan Mehmood Khan
Text lenght : 1039

is a retired two-star general of the Pakistan Army, who served as the Director General Infantry at the General Headquarters in Rawalpindi.

Education
Khan has done his PhD in Peace and Conflict Studies.

Military career
Khan was commissioned into the 34 Punjab Regiment via the 22 OTS Course. In 2018, Khan was promoted to the rank Major General from Brigadier. As Force Commander Northern Areas, he oversaw military operations and civil-military initiatives in Gilgit-Baltistan. Later he served as D

Record : 2
ID     : enwiki_80258793
Title  : Kathleen Rutherford
Text lenght : 12219

(19 June 1896 – 9 November 1975) was a Scottish-born medical practitioner, philanthropist and humanitarian aid worker. She was for many years a general practitioner (GP) in Harrogate, West Yorkshire, England. She was known locally as and .

Rutherford trained as a physician in Glasgow and Leeds, gained experience as a senior h

### Document level stats

In [22]:
num_documents = 0
total_characters = 0

min_text_length = None
max_text_length = 0

empty_documents = 0

with gzip.open(corpus_path, 'rt', encoding='utf-8') as file:

    for line in file:

        record = json.loads(line)
        text = record['text']

        text_length = len(text)

        num_documents += 1
        total_characters += text_length

        if min_text_length is None or text_length < min_text_length:
            min_text_length = text_length

        if text_length > max_text_length:
            max_text_length = text_length

average_text_length = total_characters/num_documents
print("Documents          :", f"{num_documents:,}")
print("Total characters   :", f"{total_characters:,}")
print("Average text length :", f"{average_text_length:,.2f}")
print("Minimum text length :", f"{min_text_length:,}")
print("Maximum text length :", f"{max_text_length:,}")
print("Empty documents     :", f"{empty_documents:,}")

Documents          : 229,861
Total characters   : 579,738,728
Average text length : 2,522.13
Minimum text length : 200
Maximum text length : 301,416
Empty documents     : 0


In [ ]:
document_lengths = []               # creating an empty list to store each documents length 

with gzip.open(corpus_path, "rt", encoding="utf-8") as file:

    for line in file:

        record = json.loads(line)
        text = record["text"]

        document_lengths.append(len(text))   # directly append in our empty list         


document_lengths.sort()                      # sort ascending order

num_documents = len(document_lengths)

percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]    # custom percentiles

print("Document length percentiles")
print()

for percentile in percentiles:

    index = int((percentile / 100) * (num_documents - 1))     # find appropriate list position according to that percentile
    value = document_lengths[index]                           # pull it's value

    print(f"{percentile:>2}th percentile : {value:,} characters")

Document length percentiles

 1th percentile : 212 characters
 5th percentile : 262 characters
10th percentile : 325 characters
25th percentile : 607 characters
50th percentile : 1,374 characters
75th percentile : 2,762 characters
90th percentile : 5,139 characters
95th percentile : 7,741 characters
99th percentile : 18,877 characters


In [25]:
shortest_records = []
longest_records = []

with gzip.open(corpus_path, 'rt', encoding='utf-8') as file:

    for line in file:

        record = json.loads(line)                       # converts json line into a python dictionary
        text_length = len(record['text'])

        shortest_records.append((text_length, record['title'], record['text']))     # creating a tuple containing a the length, title and full text
        longest_records.append((text_length, record['title'], record['text']))

shortest_records.sort(key=lambda x: x[0])                      # sorts by first element of each tuple which so happens to be the text length
longest_records.sort(key=lambda x: x[0], reverse=True)

print("=" * 30)
print("Shortest documents")
print("=" * 30)

for text_length, title, text in shortest_records[:5]:    # tuple unpacking and selecting the 5 shortest texts
    print()
    print("Title :", title)
    print("Length :", text_length)
    print("Text :", text[:500])

print()

print("=" * 30)
print("Longest documents")
print("=" * 30)

for text_length, title, text in longest_records[:5]:    # tuple unpacking and selecting the 5 shortest texts
    print()
    print("Title :", title)
    print("Length :", text_length)
    print("Text :", text[:500])

Shortest documents

Title : Mixtecalia
Length : 200
Text : is a genus of flowering plants in the family Asteraceae. It includes a single species, , which is endemic to Oaxaca in southern Mexico.

The genus and species were first described in 2020.

References

Title : Lungiswa
Length : 200
Text : is a given name of South African origin. Notable people with the name include:

Lungiswa Gqunta (born 1990), South African sculptor and visual artist
Lungiswa James, South African politician

See also

Title : Calosoma fabulosum
Length : 200
Text : is a species of beetle of the Carabidae family. This species is found in central eastern Afghanistan.

Adults reach a length of 26-28 mm, are brachypterous and have a reddish-brown colour.

References

Title : Clavifemora
Length : 200
Text : is an extinct genus of mesomantispine mantispid that lived during the Bathonian stage of the Middle Jurassic epoch.

Distribution
is known from the Jiulongshan Formation of Inner Mongolia.

References

Title : C

### Schema Check Consistency

In [32]:
expected_fields = {'id', 'source', 'wiki_id', 'snapshot', 'page_id', 'title', 'text'}   # python SET object since it uses {} curly braces

invalid_json_records = 0
unexpected_schema_records = 0
missing_text_records = 0
non_string_text_records = 0
total_documents = 0

with gzip.open(corpus_path, 'rt', encoding='utf-8') as file:

    for line in file:

        total_documents +=1

        try:                                           # attempts an operation that might fail
            record = json.loads(line)                  # for each line/text in the file check if all loads well using JSON
        except json.JSONDecodeError:                   # catches malformed JSON instead of crashing the loop
            invalid_json_records += 1
            continue                                   # skips the rest of the check for that bad line

        if set(record.keys()) != expected_fields:      # 'set' creates a unique set of objects and matches with expected fields we found earlier
            unexpected_schema_records += 1

        if 'text' not in record:                         # if the 'text' is not present in the record post loading json as a key
            missing_text_records += 1        

        elif not isinstance(record['text'], str):      # if text exists, checks whether it is string or not
            non_string_text_records += 1

print("Invalid JSON records       :", invalid_json_records)
print("Unexpected schema records  :", unexpected_schema_records)
print("Missing text records       :", missing_text_records)
print("Percent missing text       :", missing_text_records/total_documents)
print("Non-string text records    :", non_string_text_records)

Invalid JSON records       : 0
Unexpected schema records  : 0
Missing text records       : 0
Percent missing text       : 0.0
Non-string text records    : 0


### Text Quality Check

In [ ]:
leading_whitespaces = 0
trailing_whitespaces = 0
multiple_spaces = 0
tabs_present = 0
null_characters = 0
replacement_characters = 0

with gzip.open(corpus_path, 'rt', encoding='utf-8') as file:

    for line in file:

        record = json.loads(line)
        text = record['text']

        if text != text.lstrip():            # checks for leading whitespace/ removes the whitesapce from left
            leading_whitespaces += 1

        if text != text.rstrip():            # checks for training whitespace/removes whitespace from the right
            trailing_whitespaces += 1 

        if "  " in text:                     # checks whether there are two consecutive whitespaces anywhere, inside the quotes it is 2 spaces that we have defined
            multiple_spaces += 1

        if "\t" in text:                     # checks for tab character
            tabs_present += 1

        if "\x00" in text:                   # this is python's escapte notation fot the character whose hexadecimal value is 00
            null_characters += 1             # ord('\x00') will give us 0, because it's numeric Unicode/ASCII value is 0
                                             # eg: "hello\x00world" returns true
                                             # because the null character is present. Many times it can be invisible as well
                                             # string may visually look like 'helloworld' but internall is 'hello[NULL]world'

        if '\ufffd' in text:                 # checks for Unicode replacement character, which often indicates that some original text couldn't be decoded correctly
            replacement_characters += 1      # \ufffd is the Unicode code point for the replacement character

print("Documents with leading whitespace  :", leading_whitespaces)
print("Documents with trailing whitespace :", trailing_whitespaces)
print("Documents with multiple spaces     :", multiple_spaces)
print("Documents containing tabs          :", tabs_present)
print("Documents containing null chars    :", null_characters)
print("Documents containing �            :", replacement_characters)

Documents with leading whitespace  : 0
Documents with trailing whitespace : 0
Documents with multiple spaces     : 0
Documents containing tabs          : 0
Documents containing null chars    : 0
Documents containing �            : 0


In [37]:
## ^^ the corpus is very clean so we do not need to modify text unnecessarily.

### Train/Validation Split

In [ ]:
## 99% train & 1% validation

import random                         # python's random number generator
random.seed(42)

train_path = data_directory / 'enwiki_train.jsonl'            # a normal JSON stores all records inside one large JSON structure, often a list
                                                              # JSONL stores one complete JSON object per line
                                                              # helps store and then process one article at a time

validation_path = data_directory / 'enwiki_validation.jsonl'  # defining the train(above) and validation path

validation_probability = 0.01         # gives every document a 1% probability of going into validation

train_documents = 0
validation_documents = 0

## Open the 3 files/locations 
with (
    gzip.open(corpus_path, 'rt', encoding='utf-8') as source_file,
    open(train_path, 'w', encoding='utf-8') as train_file,
    open(validation_path, 'w', encoding='utf-8') as validation_file
):

    for line in source_file:

        record = json.loads(line)

        output_record = {                       # converts each JSON line into a dictionary with only the fields we want to keep, which are 'id', 'title' and 'text'
            'id' : record['id'],
            'title' : record['title'],
            'text'  : record['text']
        }

        output_line = json.dumps(               # converts the dictionary back into a JSON formatted string
            output_record,
            ensure_ascii=False,                 # keep actual Unicode characters in the output rather than escaping them to ASCII
        ) + '\n'                                # adds a newline character at the end of each JSON line


        if random.random() < validation_probability:   # random.random() generates a float between 0 and 1, if it is less than 0.01, it goes to validation
            validation_file.write(output_line)
            validation_documents += 1
        else:
            train_file.write(output_line)
            train_documents += 1

print("Train documents      :", f"{train_documents:,}")
print("Validation documents :", f"{validation_documents:,}")
print("Total documents      :", f"{train_documents + validation_documents:,}")

Train documents      : 227,572
Validation documents : 2,289
Total documents      : 229,861
